# Phase 1 - Model 1 Base Training on VisDrone (YOLOv8)

**Flood Victim Detection & Dynamic Rescue Priority Mapping**

Run every cell top to bottom. Expected total time: **~1.5 h on A100**, **~3.5 h on T4**.
Cell 8 asks you to set `EPOCHS` for your GPU - that is the only knob you touch.

Before you start: `Runtime > Change runtime type > GPU` (pick the best GPU you have access to).

Class scheme is `0 person, 1 vehicle, 2 animal`. `animal` is empty in VisDrone but
reserved **on purpose** so the Phase-2 flood fine-tune reuses this exact detection
head instead of reinitializing it.

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Mount Google Drive

All checkpoints go to Drive. If Colab disconnects mid-training you lose nothing -
re-run and use the resume cell (8b).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT = '/content/drive/MyDrive/FloodRescue'
RUNS    = f'{PROJECT}/runs'
MODELS  = f'{PROJECT}/models'
os.makedirs(RUNS, exist_ok=True)
os.makedirs(MODELS, exist_ok=True)
os.makedirs(f'{PROJECT}/outputs/phase1', exist_ok=True)
print(PROJECT)

## 3. Install dependencies

In [ ]:
!pip -q install "ultralytics>=8.3.0"
import ultralytics, torch
ultralytics.checks()
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 4. Download VisDrone2019-DET

Pulled from the Ultralytics asset mirror (fast, no Google-Drive quota walls).
train is ~1.4 GB, val ~0.08 GB. Kept on local Colab disk, **not** Drive - it is
regenerable, and Drive I/O would throttle every training epoch.

In [ ]:
import os
DATA = '/content/datasets/VisDrone'
os.makedirs(DATA, exist_ok=True)

BASE = 'https://github.com/ultralytics/assets/releases/download/v0.0.0'
for zipname in ['VisDrone2019-DET-train.zip', 'VisDrone2019-DET-val.zip']:
    if not os.path.isdir(f'{DATA}/{zipname[:-4]}'):
        !curl -L -o /content/{zipname} {BASE}/{zipname}
        !unzip -q /content/{zipname} -d {DATA}
        !rm /content/{zipname}

!ls {DATA}
!echo "--- train images:" && ls {DATA}/VisDrone2019-DET-train/images | wc -l
!echo "--- val images:"   && ls {DATA}/VisDrone2019-DET-val/images   | wc -l

## 5. Write the annotation converter

In [ ]:
%%writefile /content/visdrone_to_yolo.py
"""
Phase 1 - Step 1: Convert VisDrone2019-DET annotations to YOLO format.

VisDrone annotation line:
    bbox_left,bbox_top,bbox_width,bbox_height,score,category,truncation,occlusion

VisDrone categories:
    0 ignored-regions  1 pedestrian  2 people   3 bicycle  4 car   5 van
    6 truck            7 tricycle    8 awning-tricycle     9 bus  10 motor
    11 others

Project class scheme (kept IDENTICAL for the Phase-2 flood fine-tune so the
YOLO detection head is never reinitialized):
    0 person   <- pedestrian, people
    1 vehicle  <- bicycle, car, van, truck, tricycle, awning-tricycle, bus, motor
    2 animal   <- (none in VisDrone; appears only in the flood dataset)

No images are copied or moved. Labels are written to <split>/labels/ next to
<split>/images/, which is exactly where Ultralytics looks for them.

Usage:
    python visdrone_to_yolo.py --root /content/datasets/VisDrone
"""

import argparse
import json
from collections import Counter
from pathlib import Path

from PIL import Image

VISDRONE_TO_OURS = {
    1: 0, 2: 0,                                  # person
    3: 1, 4: 1, 5: 1, 6: 1, 7: 1, 8: 1, 9: 1, 10: 1,  # vehicle
}
CLASS_NAMES = ["person", "vehicle", "animal"]
SPLITS = {"train": "VisDrone2019-DET-train", "val": "VisDrone2019-DET-val"}


def convert_split(split_dir: Path, min_side_px: float = 2.0) -> dict:
    img_dir, ann_dir = split_dir / "images", split_dir / "annotations"
    lbl_dir = split_dir / "labels"
    if not img_dir.is_dir() or not ann_dir.is_dir():
        raise FileNotFoundError(f"expected images/ and annotations/ inside {split_dir}")
    lbl_dir.mkdir(exist_ok=True)

    stats = {
        "images": 0, "images_with_no_objects": 0, "boxes_kept": 0,
        "skipped_ignored_region": 0, "skipped_other_class": 0,
        "skipped_degenerate": 0, "per_class": Counter(),
        # COCO definition of a "small" object: area < 32*32 px
        "small_boxes_coco": 0, "medium_boxes_coco": 0, "large_boxes_coco": 0,
    }

    for img_path in sorted(img_dir.iterdir()):
        if img_path.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
            continue
        ann_path = ann_dir / f"{img_path.stem}.txt"
        stats["images"] += 1
        with Image.open(img_path) as im:
            W, H = im.size

        lines = []
        if ann_path.exists():
            for raw in ann_path.read_text().splitlines():
                raw = raw.strip().rstrip(",")
                if not raw:
                    continue
                p = raw.split(",")
                if len(p) < 6:
                    continue
                x, y, w, h = (float(v) for v in p[:4])
                score, cat = int(float(p[4])), int(float(p[5]))

                if score == 0 or cat == 0:          # ignored region
                    stats["skipped_ignored_region"] += 1
                    continue
                if cat not in VISDRONE_TO_OURS:     # category 11 "others"
                    stats["skipped_other_class"] += 1
                    continue

                # clamp to image bounds, then drop slivers
                x1, y1 = max(0.0, x), max(0.0, y)
                x2, y2 = min(float(W), x + w), min(float(H), y + h)
                bw, bh = x2 - x1, y2 - y1
                if bw < min_side_px or bh < min_side_px:
                    stats["skipped_degenerate"] += 1
                    continue

                cls = VISDRONE_TO_OURS[cat]
                lines.append(
                    f"{cls} {(x1 + bw / 2) / W:.6f} {(y1 + bh / 2) / H:.6f} "
                    f"{bw / W:.6f} {bh / H:.6f}"
                )
                stats["boxes_kept"] += 1
                stats["per_class"][CLASS_NAMES[cls]] += 1
                area = bw * bh
                if area < 32 ** 2:
                    stats["small_boxes_coco"] += 1
                elif area < 96 ** 2:
                    stats["medium_boxes_coco"] += 1
                else:
                    stats["large_boxes_coco"] += 1

        (lbl_dir / f"{img_path.stem}.txt").write_text("\n".join(lines))
        if not lines:
            stats["images_with_no_objects"] += 1

    stats["per_class"] = dict(stats["per_class"])
    return stats


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", required=True,
                    help="dir containing VisDrone2019-DET-train / -val")
    ap.add_argument("--yaml-out", default=None,
                    help="path for the generated data.yaml (default <root>/visdrone.yaml)")
    ap.add_argument("--stats-out", default=None,
                    help="path for the conversion stats JSON")
    args = ap.parse_args()

    root = Path(args.root).resolve()
    all_stats = {}
    for split, folder in SPLITS.items():
        print(f"[convert] {split}: {folder}")
        all_stats[split] = convert_split(root / folder)
        s = all_stats[split]
        print(f"  images={s['images']}  boxes={s['boxes_kept']}  "
              f"per_class={s['per_class']}  "
              f"small(COCO)={s['small_boxes_coco']} "
              f"({100 * s['small_boxes_coco'] / max(1, s['boxes_kept']):.1f}%)  "
              f"empty_images={s['images_with_no_objects']}")

    yaml_path = Path(args.yaml_out) if args.yaml_out else root / "visdrone.yaml"
    yaml_path.parent.mkdir(parents=True, exist_ok=True)
    yaml_path.write_text(
        "# Auto-generated by visdrone_to_yolo.py - Phase 1 base detector\n"
        f"path: {root.as_posix()}\n"
        f"train: {SPLITS['train']}/images\n"
        f"val: {SPLITS['val']}/images\n"
        "nc: 3\n"
        "names:\n"
        "  0: person\n"
        "  1: vehicle\n"
        "  2: animal   # unused in VisDrone; reserved for the Phase-2 flood fine-tune\n"
    )
    print(f"[convert] wrote {yaml_path}")

    stats_path = Path(args.stats_out) if args.stats_out else root / "visdrone_conversion_stats.json"
    stats_path.parent.mkdir(parents=True, exist_ok=True)
    stats_path.write_text(json.dumps(all_stats, indent=2))
    print(f"[convert] wrote {stats_path}")


if __name__ == "__main__":
    main()


## 6. Convert VisDrone -> YOLO format

No images are copied. Labels land in `<split>/labels/`, exactly where Ultralytics
looks for them. Also generates `visdrone.yaml` and a stats JSON - keep the stats,
the COCO *small-object* percentage is the number that justifies SAHI in Phase 3.

In [ ]:
!python /content/visdrone_to_yolo.py \
    --root {DATA} \
    --yaml-out {DATA}/visdrone.yaml \
    --stats-out {PROJECT}/outputs/phase1/visdrone_conversion_stats.json

print()
!cat {DATA}/visdrone.yaml

### 6b. Sanity check - overlay a few converted labels

If the boxes do not sit on people/cars here, **stop** and fix the conversion
before burning GPU hours.

In [ ]:
import random, glob, cv2, matplotlib.pyplot as plt

NAMES  = ['person', 'vehicle', 'animal']
COLORS = [(0, 255, 0), (0, 160, 255), (255, 0, 255)]

imgs = random.sample(sorted(glob.glob(f'{DATA}/VisDrone2019-DET-train/images/*.jpg')), 3)
fig, axes = plt.subplots(1, 3, figsize=(22, 7))
for ax, p in zip(axes, imgs):
    img = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
    H, W = img.shape[:2]
    lbl = p.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'
    n = 0
    for line in open(lbl):
        c, xc, yc, w, h = line.split()
        c = int(c); xc, yc, w, h = (float(v) for v in (xc, yc, w, h))
        x1, y1 = int((xc - w / 2) * W), int((yc - h / 2) * H)
        x2, y2 = int((xc + w / 2) * W), int((yc + h / 2) * H)
        cv2.rectangle(img, (x1, y1), (x2, y2), COLORS[c], 2)
        n += 1
    ax.imshow(img); ax.set_title(f'{os.path.basename(p)} - {n} boxes'); ax.axis('off')
plt.tight_layout(); plt.show()

## 7. Write the training script

In [ ]:
%%writefile /content/train_visdrone.py
"""
Phase 1 - Step 2: Train the Model-1 BASE detector on VisDrone (YOLOv8).

Tuned for one-shot success on Colab:
  * imgsz / batch auto-selected from the GPU actually allocated
  * mosaic kept on (big win for small objects), disabled for the last epochs
  * save_period checkpointing + --resume, so a Colab disconnect costs nothing
  * writes metrics JSON + copies best.pt to models/ for Phase 2

Usage (Colab):
    python train_visdrone.py --data /content/datasets/VisDrone/visdrone.yaml \
                             --project /content/drive/MyDrive/FloodRescue/runs
    python train_visdrone.py ... --resume      # after a disconnect
"""

import argparse
import json
import shutil
from pathlib import Path

import torch
from ultralytics import YOLO

# GPU-name substring -> (imgsz, batch).  Fallback used for anything unlisted.
GPU_PRESETS = [
    ("a100", (1280, 16)),
    ("h100", (1280, 24)),
    ("l4",   (1024, 10)),
    ("v100", (1024, 10)),
    ("t4",   ( 960, 12)),
    ("4090", (1280, 12)),
]
FALLBACK = (1024, 8)


def pick_preset():
    if not torch.cuda.is_available():
        raise SystemExit("No CUDA GPU. In Colab: Runtime > Change runtime type > GPU.")
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    low = name.lower()
    for key, preset in GPU_PRESETS:
        if key in low:
            print(f"[gpu] {name} ({vram:.0f} GB) -> imgsz={preset[0]} batch={preset[1]}")
            return preset
    print(f"[gpu] {name} ({vram:.0f} GB) unlisted -> imgsz={FALLBACK[0]} batch={FALLBACK[1]}")
    return FALLBACK


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True, help="path to visdrone.yaml")
    ap.add_argument("--model", default="yolov8s.pt")
    ap.add_argument("--project", default="runs/phase1",
                    help="put this on Google Drive so checkpoints survive a disconnect")
    ap.add_argument("--name", default="visdrone_base")
    ap.add_argument("--epochs", type=int, default=60)
    ap.add_argument("--patience", type=int, default=15,
                    help="epochs without mAP improvement before early stop")
    ap.add_argument("--imgsz", type=int, default=0, help="0 = auto from GPU")
    ap.add_argument("--batch", type=int, default=0, help="0 = auto from GPU")
    ap.add_argument("--workers", type=int, default=8)
    ap.add_argument("--resume", action="store_true")
    ap.add_argument("--export-to", default="models",
                    help="dir to copy best.pt into as visdrone_base.pt")
    args = ap.parse_args()

    auto_imgsz, auto_batch = pick_preset()
    imgsz = args.imgsz or auto_imgsz
    batch = args.batch or auto_batch

    run_dir = Path(args.project) / args.name
    last = run_dir / "weights" / "last.pt"

    if args.resume:
        if not last.exists():
            raise SystemExit(f"--resume given but {last} not found")
        print(f"[resume] {last}")
        model = YOLO(str(last))
        results = model.train(resume=True)
    else:
        model = YOLO(args.model)
        results = model.train(
            data=args.data,
            epochs=args.epochs,
            imgsz=imgsz,
            batch=batch,
            device=0,
            workers=args.workers,
            project=args.project,
            name=args.name,
            exist_ok=True,
            # --- optimisation ---
            optimizer="SGD",
            lr0=0.01,
            lrf=0.01,
            cos_lr=True,
            momentum=0.937,
            weight_decay=0.0005,
            warmup_epochs=3.0,
            patience=args.patience,
            # --- augmentation: biased toward small, dense aerial targets ---
            mosaic=1.0,
            close_mosaic=10,      # last 10 epochs clean, so val-time stats match
            scale=0.5,
            translate=0.1,
            fliplr=0.5,
            flipud=0.0,           # drone imagery is gravity-oriented
            degrees=0.0,
            hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
            erasing=0.0,
            # --- runtime ---
            amp=True,
            cache=False,          # VisDrone at 1024+ will not fit in Colab RAM
            save_period=5,        # checkpoint every 5 epochs
            plots=True,
            seed=0,
            val=True,
        )

    save_dir = Path(results.save_dir if hasattr(results, "save_dir") else run_dir)
    best = save_dir / "weights" / "best.pt"
    print(f"[train] done -> {best}")

    # ---- final validation at the trained resolution -------------------------
    print("[val] running final validation on the VisDrone val split")
    metrics = YOLO(str(best)).val(
        data=args.data, imgsz=imgsz, batch=batch, device=0,
        project=args.project, name=f"{args.name}_val", exist_ok=True, plots=True,
    )
    box = metrics.box
    summary = {
        "weights": str(best),
        "imgsz": imgsz,
        "batch": batch,
        "mAP50-95": round(float(box.map), 4),
        "mAP50": round(float(box.map50), 4),
        "mAP75": round(float(box.map75), 4),
        "precision": round(float(box.mp), 4),
        "recall": round(float(box.mr), 4),
        "per_class_mAP50-95": {
            metrics.names[c]: round(float(box.maps[c]), 4) for c in range(len(box.maps))
        },
    }
    print(json.dumps(summary, indent=2))

    out = save_dir / "phase1_visdrone_metrics.json"
    out.write_text(json.dumps(summary, indent=2))
    print(f"[val] wrote {out}")

    if args.export_to:
        dest_dir = Path(args.export_to)
        dest_dir.mkdir(parents=True, exist_ok=True)
        shutil.copy2(best, dest_dir / "visdrone_base.pt")
        shutil.copy2(out, dest_dir / "phase1_visdrone_metrics.json")
        print(f"[export] {dest_dir / 'visdrone_base.pt'}  <-- feed this to Phase 2")


if __name__ == "__main__":
    main()


## 8. Train

`imgsz` and `batch` are picked automatically from whichever GPU Colab handed you
(1280 on A100, 1024 on L4/V100, 960 on T4). A checkpoint is written to Drive
every 5 epochs, and `patience` stops early once mAP plateaus.

**Set `EPOCHS` below to match your GPU** - cell 1 told you which one you got:

| GPU | EPOCHS | PATIENCE | ~total |
|---|---|---|---|
| A100 | 60 | 15 | ~1.5 h |
| L4 | 50 | 12 | ~3.5 h |
| **T4** | **35** | **10** | **~3.5 h** |

On a T4, 60 epochs would run 8-10 h and will not survive one free Colab session.
35 epochs from COCO-pretrained weights is ample for a *base* model that Phase 2
fine-tunes anyway - over-optimizing Stage 1 is wasted GPU time.

Leave this tab open and the browser awake: a free Colab session is reclaimed
after roughly 90 minutes of no interaction.

In [ ]:
EPOCHS   = 35   # T4: 35  |  L4: 50  |  A100: 60
PATIENCE = 10   # T4: 10  |  L4: 12  |  A100: 15

!python /content/train_visdrone.py \
    --data {DATA}/visdrone.yaml \
    --model yolov8s.pt \
    --project {RUNS} \
    --name visdrone_base \
    --epochs {EPOCHS} \
    --patience {PATIENCE} \
    --export-to {MODELS}

### 8b. If Colab disconnected - resume

Re-run cells 1-7 (download and convert are idempotent and skip if already done),
then run this **instead of** cell 8.

In [ ]:
!python /content/train_visdrone.py \
    --data {DATA}/visdrone.yaml \
    --project {RUNS} \
    --name visdrone_base \
    --resume \
    --export-to {MODELS}

## 9. Results - the numbers for your report

In [ ]:
import json
from IPython.display import Image as IPImage, display

m = json.load(open(f'{MODELS}/phase1_visdrone_metrics.json'))
print(json.dumps(m, indent=2))

for png in ['results.png', 'confusion_matrix_normalized.png', 'PR_curve.png',
            'val_batch0_pred.jpg']:
    p = f'{RUNS}/visdrone_base/{png}'
    if os.path.exists(p):
        print('\n' + png)
        display(IPImage(p, width=900))

### Sanity targets for `yolov8s` on VisDrone (2 merged classes)

| metric | A100 @1280, 60ep | **T4 @960, 35ep** |
|---|---|---|
| mAP@50 | 0.46 - 0.52 | **0.40 - 0.48** |
| mAP@50-95 | 0.27 - 0.31 | **0.23 - 0.28** |
| person mAP@50 | 0.38 - 0.45 | 0.32 - 0.40 |
| vehicle mAP@50 | 0.54 - 0.62 | 0.48 - 0.57 |

The T4 column is 2-4 points lower - that is the cost of 960 px and 35 epochs,
and it is the right trade. Phase 2 fine-tuning and Phase 3 SAHI both move the
number far more than those points would.

`animal` will read **0.0** - correct, VisDrone has no animal instances. Landing
well below this range means something went wrong in conversion, not in training.

## 10. Inference spot-check on val images

In [ ]:
from ultralytics import YOLO
import glob, random, matplotlib.pyplot as plt, cv2

model = YOLO(f'{MODELS}/visdrone_base.pt')
picks = random.sample(sorted(glob.glob(f'{DATA}/VisDrone2019-DET-val/images/*.jpg')), 3)
res = model.predict(picks, imgsz=m['imgsz'], conf=0.25, verbose=False)

fig, axes = plt.subplots(1, 3, figsize=(22, 7))
for ax, r in zip(axes, res):
    ax.imshow(cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB))
    ax.set_title(f'{len(r.boxes)} detections'); ax.axis('off')
plt.tight_layout(); plt.show()

## 11. Download for the local repo

Put `visdrone_base.pt` into `E:/Flood_major_7/models/` - Phase 2 fine-tunes from it.

In [ ]:
from google.colab import files
files.download(f'{MODELS}/visdrone_base.pt')
files.download(f'{MODELS}/phase1_visdrone_metrics.json')

---
### Phase 1 deliverables checklist

- [ ] `models/visdrone_base.pt` - base detector weights (input to Phase 2)
- [ ] `models/phase1_visdrone_metrics.json` - the VisDrone-only baseline mAP
      (**Phase 9 compares the fine-tuned model against this**)
- [ ] `outputs/phase1/visdrone_conversion_stats.json` - class counts + COCO small/medium/large split
- [ ] `runs/visdrone_base/` - results.png, PR_curve.png, confusion_matrix_normalized.png

Next: **Phase 2** - fine-tune this on the Roboflow flood dataset at a lower LR.